# 모듈을 사용하는 작업 시작 전 평가

계산·저장·조회 함수는 risk_engine.py에서 관리합니다.
맨 아래에 평가 기록 만들기, 저장 및 다시 조회, 저장된 평가 검색, 평가ID 선택 셀이 있습니다.
모듈 변경 후에는 커널을 재시작하고 위에서부터 실행하세요. 등급은 잠정입니다.


In [31]:
from pathlib import Path
import sys
import pandas as pd
candidates = [
    candidate
    for parent in [Path.cwd(), *Path.cwd().parents]
    for candidate in [parent, parent / "fire_risk_project"]
    if (candidate / "risk_engine.py").is_file()
]
if not candidates:
    raise FileNotFoundError("fire_risk_project 또는 notebooks 폴더에서 실행하세요.")
project_dir = candidates[0]
sys.path.insert(0, str(project_dir))
import risk_engine as engine
print("사용하는 계산 모듈:", engine.__file__)


사용하는 계산 모듈: c:\Users\uijun\OneDrive\바탕 화면\Project\fire_risk_project\risk_engine.py


In [32]:
with pd.option_context('display.max_colwidth', 100):
    display(engine.watcher_questions)


,추가질문ID,구분,질문,허용응답
0,W001,장소조건,작업반경 11m 이내 건물구조 자체 또는 내부(개구부 등으로 개방된 부분 포함)에 가연성물질이 있는가?,예 / 아니오 / 확인불가
1,W002,장소조건,작업반경 11m 이내의 바닥 하부에 가연성물질이 11m 이상 떨어져 있지만 불꽃에 의해 쉽게 발화될 우려가 있는가?,예 / 아니오 / 확인불가
2,W003,장소조건,금속 칸막이·벽·천장·지붕의 반대쪽 면에 가연성물질이 인접하여 열전도·열복사로 발화될 우려가 있는가?,예 / 아니오 / 확인불가
3,W004,예외검토,같은 장소에서 상시·반복적으로 용접·용단하는 경우의 설비 구비 등 법령상 배치 예외를 적용하려는가?,예 / 아니오 / 확인불가


In [33]:
watcher_answers = {
    "W001": "예", "W002": "아니오",
    "W003": "아니오", "W004": "아니오",
}
deployment_answer = "미배치"  # 가상 예제
watcher_check, watcher_detail = engine.check_h05_watcher_response(
    watcher_answers, deployment_answer
)
display(pd.DataFrame(watcher_check.items(), columns=["항목", "결과"]))


,항목,결과
0,작업ID,H05
1,대상항목ID,Q004
2,연결기준ID,R002
3,장소조건검토,장소조건 해당 응답 있음
4,해당조건ID,W001
5,미확인조건ID,-
6,검토결과,"배치 대상 검토: 장소조건 해당, 예외 적용 요청 없음"
7,해당없음자동승인,False
8,Q004응답,미배치
9,응답비교결과,배치 상태 확인 필요


## 화재감시자 분기 검사
원본 노트북의 6개 사례를 그대로 실행합니다.


In [34]:
test_cases = [
    (
        "조건 해당 + 미배치",
        {"W001": "예", "W002": "아니오",
         "W003": "아니오", "W004": "아니오"},
        "미배치",
        "배치 상태 확인 필요",
    ),
    (
        "조건 해당 + 배치",
        {"W001": "예", "W002": "아니오",
         "W003": "아니오", "W004": "아니오"},
        "배치",
        "배치 응답 확인",
    ),
    (
        "현장 조건 미확인",
        {"W001": "확인불가", "W002": "아니오",
         "W003": "아니오", "W004": "아니오"},
        "미배치",
        "현장 조건 확인 필요",
    ),
    (
        "배치 예외 검토 요청",
        {"W001": "예", "W002": "아니오",
         "W003": "아니오", "W004": "예"},
        "미배치",
        "예외 적용 검토 필요",
    ),
    (
        "해당없음 자동 승인 방지",
        {"W001": "아니오", "W002": "아니오",
         "W003": "아니오", "W004": "아니오"},
        "해당없음",
        "해당없음 근거 검토 필요",
    ),
    (
        "모든 장소조건 비해당",
        {"W001": "아니오", "W002": "아니오",
         "W003": "아니오", "W004": "아니오"},
        "미배치",
        "적용 여부 근거 확인 필요",
    ),
]

test_results = []

for name, answers, deployment, expected in test_cases:
    result, _ = engine.check_h05_watcher_response(answers, deployment)
    actual = result["응답비교결과"]

    test_results.append({
        "테스트": name,
        "예상결과": expected,
        "실제결과": actual,
        "통과": actual == expected,
    })

test_table = pd.DataFrame(test_results)
display(test_table)

assert test_table["통과"].all(), "실패한 테스트의 결과를 확인하세요."
print(f"총 {len(test_table)}개 테스트 통과")

,테스트,예상결과,실제결과,통과
0,조건 해당 + 미배치,배치 상태 확인 필요,배치 상태 확인 필요,True
1,조건 해당 + 배치,배치 응답 확인,배치 응답 확인,True
2,현장 조건 미확인,현장 조건 확인 필요,현장 조건 확인 필요,True
3,배치 예외 검토 요청,예외 적용 검토 필요,예외 적용 검토 필요,True
4,해당없음 자동 승인 방지,해당없음 근거 검토 필요,해당없음 근거 검토 필요,True
5,모든 장소조건 비해당,적용 여부 근거 확인 필요,적용 여부 근거 확인 필요,True


총 6개 테스트 통과


## 전체 위험점수 계산
data 폴더의 DB와 사전평가 설정을 읽습니다.


In [35]:
if not (project_dir / 'data/09_사전평가설정.csv').is_file():
    print('전체 점수 계산 보류: data/09_사전평가설정.csv를 복원하세요.')
else:
    engine.load_databases(project_dir / 'data')
    h05_answers = {
        "Q001": "있음",
        "Q002": 5,
        "Q003": "미실시",
        "Q004": "미배치",
        "Q018": "용접방화포 미사용",
        "Q019": "없음",
        "Q020": "없음",
        "Q025": "적정",
    }
    summary, detail = engine.calculate_task_risk("H05", h05_answers)
    display(pd.DataFrame([summary]))
    display(detail)
    display(engine.get_top_risks(detail))
    
    prework_question_table = engine.get_prework_questions("H05", h05_answers)
    display(prework_question_table[["항목ID", "확인항목", "적용상태", "제외사유"]])

DB의 기본 연결·중복·집계 설정 검사 통과
사전평가 설정 검사 통과


,작업ID,평가시점,적용질문수,제외질문수,입력질문수,집계항목수,총가중위험점수,최대가중점수,위험지수,기본위험등급(잠정),강제경고,최종판정,계산상태
0,H05,작업시작전,8,1,8,7,13.5,30.0,45.0,주의,미검증,보류: 법령·잔여 적용조건 미검증,사전 시험 계산


,항목ID,확인항목,사용자응답,단위,점수규칙ID,조건순번,점수,가중치,가중점수,최대가중점수,...,그룹집계방식,관련기준ID,적용조건,확정상태,평가시점,적용상태,제외사유,사전안내,응답상태,집계대표여부
0,Q001,주변 가연물 존재 여부,있음,-,S001,1,1.0,1.5,1.5,4.5,...,MAX_PER_TASK,R001,화기·불티/전기/열원 작업 공통,미확정,작업시작전,적용,,작업 시작 직전에 확인한 현장 상태로 응답합니다. 미확인 상태를 안전한 응답으로 추...,입력,True
1,Q002,가연물과 작업점 거리,5.0,m,S002,1,1.0,1.5,1.5,4.5,...,MAX_PER_TASK,R001;R002,화기·불티 작업,미확정,작업시작전,적용,,작업 시작 직전에 확인한 현장 상태로 응답합니다. 미확인 상태를 안전한 응답으로 추...,입력,True
2,Q003,불티 비산방지조치(용접방화포 등) 실시 여부,미실시,-,S003,1,3.0,1.5,4.5,4.5,...,MAX_PER_TASK,R001,화기·불티 작업,미확정,작업시작전,적용,,작업 시작 직전에 확인한 현장 상태로 응답합니다. 미확인 상태를 안전한 응답으로 추...,입력,True
3,Q004,화재감시자 배치 여부,미배치,-,S004,1,3.0,2.0,6.0,6.0,...,MAX_PER_TASK,R002,법령상 배치대상 조건 충족 시 적용,미확정,작업시작전,적용,,작업 시작 직전에 확인한 현장 상태로 응답합니다. 미확인 상태를 안전한 응답으로 추...,입력,True
4,Q018,용접방화포 사용 시 성능인증 제품 여부,용접방화포 미사용,-,S019,3,0.0,0.0,0.0,0.0,...,MAX_PER_TASK,R001,다른 적정 비산방지조치를 적용하는 경우,미확정,작업시작전,적용,,작업 시작 직전에 확인한 현장 상태로 응답합니다. 미확인 상태를 안전한 응답으로 추...,입력,False
5,Q019,작업점 하부 가연물의 불티 착화 우려 여부,없음,-,S020,2,0.0,1.0,0.0,3.0,...,MAX_PER_TASK,R002,다른 배치조건도 함께 확인,미확정,작업시작전,적용,,작업 시작 직전에 확인한 현장 상태로 응답합니다. 미확인 상태를 안전한 응답으로 추...,입력,True
6,Q020,금속 칸막이·벽·천장·지붕 반대면 가연물의 열전도·복사 착화 우려 여부,없음,-,S021,2,0.0,1.0,0.0,3.0,...,MAX_PER_TASK,R002,다른 배치조건도 함께 확인,미확정,작업시작전,적용,,작업 시작 직전에 확인한 현장 상태로 응답합니다. 미확인 상태를 안전한 응답으로 추...,입력,True
7,Q025,초기소화설비 비치·접근 가능 여부,적정,-,S026,1,0.0,1.5,0.0,4.5,...,MAX_PER_TASK,R001;R014,화재위험 작업,미확정,작업시작전,적용,,작업 시작 직전에 확인한 현장 상태로 응답합니다. 미확인 상태를 안전한 응답으로 추...,입력,True
8,Q026,작업 종료 후 잔불·불티 확인 여부,None,-,S027,,NaN,NaN,0.0,0.0,...,MAX_PER_TASK,R015,,점수미산정,작업시작전,시점제외,작업 종료 후 잔불·불티 확인 항목으로 사전 점수 합산에서 제외합니다. 종료 후 확...,작업 종료 후 잔불·불티 확인 항목으로 사전 점수 합산에서 제외합니다. 종료 후 확...,미응답,False


,항목ID,확인항목,사용자응답,가중점수,관련기준ID
0,Q004,화재감시자 배치 여부,미배치,6.0,R002
1,Q003,불티 비산방지조치(용접방화포 등) 실시 여부,미실시,4.5,R001
2,Q001,주변 가연물 존재 여부,있음,1.5,R001


,항목ID,확인항목,적용상태,제외사유
0,Q001,주변 가연물 존재 여부,적용,
1,Q002,가연물과 작업점 거리,적용,
2,Q003,불티 비산방지조치(용접방화포 등) 실시 여부,적용,
3,Q004,화재감시자 배치 여부,적용,
4,Q018,용접방화포 사용 시 성능인증 제품 여부,적용,
5,Q019,작업점 하부 가연물의 불티 착화 우려 여부,적용,
6,Q020,금속 칸막이·벽·천장·지붕 반대면 가연물의 열전도·복사 착화 우려 여부,적용,
7,Q025,초기소화설비 비치·접근 가능 여부,적용,
8,Q026,작업 종료 후 잔불·불티 확인 여부,시점제외,작업 종료 후 잔불·불티 확인 항목으로 사전 점수 합산에서 제외합니다. 종료 후 확...


## 평가 기록 만들기
현장명과 앞서 입력한 h05_answers, watcher_answers를 같은 현장 기준으로 맞추세요. Q004는 h05_answers에서 가져와 재계산합니다.

평가ID·일시는 같은 커널에서 유지됩니다. 입력을 바꿔 새 평가로 저장하려면 먼저 evaluation_meta = None을 실행하세요. 기존 평가의 덮어쓰기는 지원하지 않습니다.


In [36]:
from datetime import datetime, timezone, timedelta
from uuid import uuid4

site_name = "시험 현장"  # 실제 사용 시 현장명 수정
if globals().get("evaluation_meta") is None:
    evaluation_meta = {
        "evaluation_id": str(uuid4()),
        "evaluated_at": datetime.now(
            timezone(timedelta(hours=9))
        ).isoformat(timespec="seconds"),
    }

evaluation = engine.build_evaluation(
    site_name=site_name,
    task_id="H05",
    answers=h05_answers,
    watcher_answers=watcher_answers,
    **evaluation_meta,
)
display(pd.DataFrame([{
    "평가ID": evaluation["평가ID"],
    "현장명": evaluation["현장명"],
    **evaluation["요약"],
}]))
display(pd.DataFrame([evaluation["화재감시자검토"]]))


,평가ID,현장명,작업ID,평가시점,적용질문수,제외질문수,입력질문수,집계항목수,총가중위험점수,최대가중점수,위험지수,기본위험등급(잠정),강제경고,최종판정,계산상태
0,c7279531-35bf-4990-a8ba-c1a918871db1,시험 현장,H05,작업시작전,8,1,8,7,13.5,30.0,45.0,주의,미검증,보류: 법령·잔여 적용조건 미검증,사전 시험 계산


,작업ID,대상항목ID,연결기준ID,장소조건검토,해당조건ID,미확인조건ID,검토결과,해당없음자동승인,Q004응답,응답비교결과,확인메시지,최종판정
0,H05,Q004,R002,장소조건 해당 응답 있음,W001,-,"배치 대상 검토: 장소조건 해당, 예외 적용 요청 없음",False,미배치,배치 상태 확인 필요,"장소조건에 해당하는 응답이 있고 예외 적용 요청은 없지만, 화재감시자가 미배치로 입...",보류: 현장 근거 및 나머지 기준 미검증


## 저장 및 다시 조회
현재 경로는 시험용 저장소입니다. 먼저 결과를 확인한 뒤 아래 save_enabled를 True로 바꾸고 이 셀을 실행하세요. 같은 기록의 반복 저장은 중복되지 않습니다. 실제 기록은 별도 경로로 구분하세요. 기존 06번 CSV에는 기록하지 않습니다.


In [37]:
storage_path = project_dir / "outputs" / "evaluations_demo.sqlite3"
save_enabled = False  # 시험 저장하려면 True로 변경

if save_enabled:
    print(engine.save_evaluation(evaluation, storage_path))
    saved = engine.load_evaluation(evaluation["평가ID"], storage_path)
    assert saved == evaluation, "저장 전후 기록이 다릅니다."
    display(engine.list_evaluations(storage_path))
    display(pd.DataFrame(saved["응답상세"]))
    display(pd.DataFrame(saved["화재감시자응답"]))
else:
    print("아직 저장하지 않았습니다. 위 결과를 확인하고 save_enabled = True로 변경하세요.")


아직 저장하지 않았습니다. 위 결과를 확인하고 save_enabled = True로 변경하세요.


## 저장된 평가 검색
아래 두 조회 셀만 실행해도 됩니다(처음에는 맨 위 모듈 불러오기 셀 실행). 저장 버튼을 다시 실행할 필요가 없습니다. 현장명은 일부 글자로 검색하고 작업ID는 정확하게 입력하세요. 빈 문자열은 전체 조회입니다.


In [38]:
history_path = project_dir / "outputs" / "evaluations_demo.sqlite3"
site_keyword = ""  # 예: "시험" 또는 실제 현장명 일부
task_filter = ""   # 예: "H05". 빈 문자열이면 모든 작업

history = engine.search_evaluations(
    history_path, site_keyword=site_keyword, task_id=task_filter
)
if history.empty:
    print("조회된 평가가 없습니다. 저장 여부, 저장소 경로와 검색 조건을 확인하세요.")
else:
    display(history)


,평가ID,평가일시,현장명,작업ID,위험지수,기본위험등급(잠정),최종판정
0,c7279531-35bf-4990-a8ba-c1a918871db1,2026-10-11T17:25:44+09:00,시험 현장,H05,45.0,주의,보류: 법령·잔여 적용조건 미검증


## 평가ID를 선택해 당시 기록 보기
목록의 평가ID를 복사해 아래 문자열에 넣으세요. 빈 문자열이면 검색 결과 중 가장 최근 평가를 표시합니다. 당시 입력·계산 결과를 읽으며 현재 CSV로 재계산하지 않습니다. 최종판정 보류 상태도 함께 표시합니다.


In [39]:
selected_evaluation_id = ""  # 보고 싶은 평가ID. 비워 두면 검색 결과의 최근 기록
# 검색 조건이 바뀌었을 수 있으므로 조회할 때 목록을 갱신합니다.
history = engine.search_evaluations(
    history_path, site_keyword=site_keyword, task_id=task_filter
)
if history.empty:
    print("선택할 평가가 없습니다. 먼저 시험 평가를 한 건 저장하세요.")
else:
    chosen_id = selected_evaluation_id.strip() or history.iloc[0]["평가ID"]
    if chosen_id not in set(history["평가ID"]):
        print("현재 검색 결과에 없는 평가ID입니다. 검색 조건이나 평가ID를 확인하세요.")
    else:
        saved_tables = engine.get_evaluation_tables(chosen_id, history_path)
        print("조회한 평가ID:", chosen_id)
        for title, table in saved_tables.items():
            print(title)
            if table.empty:
                print("기록 없음")
            else:
                with pd.option_context("display.max_colwidth", 120):
                    display(table)


조회한 평가ID: c7279531-35bf-4990-a8ba-c1a918871db1
평가요약


,평가ID,평가일시,현장명,작업ID,강제경고,계산상태,기본위험등급(잠정),위험지수,입력질문수,적용질문수,제외질문수,집계항목수,총가중위험점수,최대가중점수,최종판정,평가시점
0,c7279531-35bf-4990-a8ba-c1a918871db1,2026-10-11T17:25:44+09:00,시험 현장,H05,미검증,사전 시험 계산,주의,45.0,8,8,1,7,13.5,30.0,보류: 법령·잔여 적용조건 미검증,작업시작전


원래입력


,항목ID,입력값
0,Q001,있음
1,Q002,5
2,Q003,미실시
3,Q004,미배치
4,Q018,용접방화포 미사용
5,Q019,없음
6,Q020,없음
7,Q025,적정


응답상세


,가중점수,가중치,관련기준ID,그룹집계방식,단위,사용자응답,사전안내,응답상태,적용상태,적용조건,...,점수합산대상,제외사유,조건순번,중복집계그룹,집계대표여부,최대가중점수,평가시점,항목ID,확인항목,확정상태
0,1.5,1.5,R001,MAX_PER_TASK,-,있음,작업 시작 직전에 확인한 현장 상태로 응답합니다. 미확인 상태를 안전한 응답으로 추정하지 않습니다.,입력,적용,화기·불티/전기/열원 작업 공통,...,True,,1,Q001,True,4.5,작업시작전,Q001,주변 가연물 존재 여부,미확정
1,1.5,1.5,R001;R002,MAX_PER_TASK,m,5.0,작업 시작 직전에 확인한 현장 상태로 응답합니다. 미확인 상태를 안전한 응답으로 추정하지 않습니다.,입력,적용,화기·불티 작업,...,True,,1,Q002,True,4.5,작업시작전,Q002,가연물과 작업점 거리,미확정
2,4.5,1.5,R001,MAX_PER_TASK,-,미실시,작업 시작 직전에 확인한 현장 상태로 응답합니다. 미확인 상태를 안전한 응답으로 추정하지 않습니다. 해당없음의 타당성과 법령 적용조건은 별도로 검토해야 합니다.,입력,적용,화기·불티 작업,...,True,,1,Q003,True,4.5,작업시작전,Q003,불티 비산방지조치(용접방화포 등) 실시 여부,미확정
3,6.0,2.0,R002,MAX_PER_TASK,-,미배치,작업 시작 직전에 확인한 현장 상태로 응답합니다. 미확인 상태를 안전한 응답으로 추정하지 않습니다. 해당없음의 타당성과 법령 적용조건은 별도로 검토해야 합니다.,입력,적용,법령상 배치대상 조건 충족 시 적용,...,True,,1,Q004,True,6.0,작업시작전,Q004,화재감시자 배치 여부,미확정
4,0.0,0.0,R001,MAX_PER_TASK,-,용접방화포 미사용,작업 시작 직전에 확인한 현장 상태로 응답합니다. 미확인 상태를 안전한 응답으로 추정하지 않습니다.,입력,적용,다른 적정 비산방지조치를 적용하는 경우,...,False,,3,Q018,False,0.0,작업시작전,Q018,용접방화포 사용 시 성능인증 제품 여부,미확정
5,0.0,1.0,R002,MAX_PER_TASK,-,없음,작업 시작 직전에 확인한 현장 상태로 응답합니다. 미확인 상태를 안전한 응답으로 추정하지 않습니다.,입력,적용,다른 배치조건도 함께 확인,...,True,,2,Q019,True,3.0,작업시작전,Q019,작업점 하부 가연물의 불티 착화 우려 여부,미확정
6,0.0,1.0,R002,MAX_PER_TASK,-,없음,작업 시작 직전에 확인한 현장 상태로 응답합니다. 미확인 상태를 안전한 응답으로 추정하지 않습니다.,입력,적용,다른 배치조건도 함께 확인,...,True,,2,Q020,True,3.0,작업시작전,Q020,금속 칸막이·벽·천장·지붕 반대면 가연물의 열전도·복사 착화 우려 여부,미확정
7,0.0,1.5,R001;R014,MAX_PER_TASK,-,적정,작업 시작 직전에 확인한 현장 상태로 응답합니다. 미확인 상태를 안전한 응답으로 추정하지 않습니다.,입력,적용,화재위험 작업,...,True,,1,Q025,True,4.5,작업시작전,Q025,초기소화설비 비치·접근 가능 여부,미확정
8,0.0,NaN,R015,MAX_PER_TASK,-,None,작업 종료 후 잔불·불티 확인 항목으로 사전 점수 합산에서 제외합니다. 종료 후 확인은 별도로 수행합니다.,미응답,시점제외,,...,False,작업 종료 후 잔불·불티 확인 항목으로 사전 점수 합산에서 제외합니다. 종료 후 확인은 별도로 수행합니다.,,Q026,False,0.0,작업시작전,Q026,작업 종료 후 잔불·불티 확인 여부,점수미산정


화재감시자검토


,Q004응답,검토결과,대상항목ID,미확인조건ID,연결기준ID,응답비교결과,작업ID,장소조건검토,최종판정,해당없음자동승인,해당조건ID,확인메시지
0,미배치,"배치 대상 검토: 장소조건 해당, 예외 적용 요청 없음",Q004,-,R002,배치 상태 확인 필요,H05,장소조건 해당 응답 있음,보류: 현장 근거 및 나머지 기준 미검증,False,W001,"장소조건에 해당하는 응답이 있고 예외 적용 요청은 없지만, 화재감시자가 미배치로 입력되어 있습니다."


추가질문응답


,구분,응답,질문,추가질문ID,허용응답
0,장소조건,예,작업반경 11m 이내 건물구조 자체 또는 내부(개구부 등으로 개방된 부분 포함)에 가연성물질이 있는가?,W001,예 / 아니오 / 확인불가
1,장소조건,아니오,작업반경 11m 이내의 바닥 하부에 가연성물질이 11m 이상 떨어져 있지만 불꽃에 의해 쉽게 발화될 우려가 있는가?,W002,예 / 아니오 / 확인불가
2,장소조건,아니오,금속 칸막이·벽·천장·지붕의 반대쪽 면에 가연성물질이 인접하여 열전도·열복사로 발화될 우려가 있는가?,W003,예 / 아니오 / 확인불가
3,예외검토,아니오,같은 장소에서 상시·반복적으로 용접·용단하는 경우의 설비 구비 등 법령상 배치 예외를 적용하려는가?,W004,예 / 아니오 / 확인불가


In [40]:
import importlib

print("불러온 파일:", engine.__file__)

engine = importlib.reload(engine)
engine.load_databases(project_dir / "data")

print("비교 함수 있음:", hasattr(engine, "compare_evaluations"))

불러온 파일: c:\Users\uijun\OneDrive\바탕 화면\Project\fire_risk_project\risk_engine.py
DB의 기본 연결·중복·집계 설정 검사 통과
사전평가 설정 검사 통과
비교 함수 있음: True


## 두 평가 비교 연습
아래는 Q004만 미배치에서 배치로 바꾼 가상 예제입니다. 실제 현장 기록이 아닙니다. 임시 저장소를 사용하므로 기존 평가 목록에는 추가되지 않습니다. 예상 점수는 45 → 25, 차이는 -20입니다.


In [41]:
from tempfile import TemporaryDirectory

comparison_before_answers = dict(h05_answers)
comparison_before_answers.update({
    "Q001": "있음", "Q002": 5, "Q003": "미실시", "Q004": "미배치",
    "Q018": "용접방화포 미사용", "Q019": "없음", "Q020": "없음", "Q025": "적정",
})
comparison_after_answers = {**comparison_before_answers, "Q004": "배치"}
comparison_extra = {"W001": "예", "W002": "아니오", "W003": "아니오", "W004": "아니오"}

with TemporaryDirectory() as comparison_temp:
    comparison_db = Path(comparison_temp) / "comparison.sqlite3"
    for example_id, answers, stamp in [
        ("example-before", comparison_before_answers, "2026-10-11T09:00:00+09:00"),
        ("example-after", comparison_after_answers, "2026-10-11T10:00:00+09:00"),
    ]:
        example = engine.build_evaluation(
            "비교 연습용 가상 현장", "H05", answers, comparison_extra,
            evaluation_id=example_id, evaluated_at=stamp,
        )
        engine.save_evaluation(example, comparison_db)
    comparison_example = engine.compare_evaluations(
        "example-before", "example-after", comparison_db
    )

display(pd.DataFrame([comparison_example["비교안내"]]))
display(comparison_example["평가요약"])
display(comparison_example["변경항목"])
display(comparison_example["화재감시자검토"])


,점수차이표시가능,위험지수차이(이후-이전),비교제한사유,해석
0,True,-20.0,없음,저장 당시의 시험 점수 비교입니다. 점수 하락만으로 안전 또는 조치 완료를 판정하지...


,구분,평가ID,평가일시,현장명,작업ID,위험지수,최대가중점수,기본위험등급(잠정),최종판정
0,이전,example-before,2026-10-11T09:00:00+09:00,비교 연습용 가상 현장,H05,45.0,30.0,주의,보류: 법령·잔여 적용조건 미검증
1,이후,example-after,2026-10-11T10:00:00+09:00,비교 연습용 가상 현장,H05,25.0,30.0,주의,보류: 법령·잔여 적용조건 미검증


,항목ID,확인항목,이전_사용자응답,이후_사용자응답,이전_가중점수,이후_가중점수,이전_최대가중점수,이후_최대가중점수,이전_적용상태,이후_적용상태,이전_집계대표여부,이후_집계대표여부
0,Q004,화재감시자 배치 여부,미배치,배치,6.0,0.0,6.0,6.0,적용,적용,True,True


,구분,Q004응답,검토결과,대상항목ID,미확인조건ID,연결기준ID,응답비교결과,작업ID,장소조건검토,최종판정,해당없음자동승인,해당조건ID,확인메시지
0,이전,미배치,"배치 대상 검토: 장소조건 해당, 예외 적용 요청 없음",Q004,-,R002,배치 상태 확인 필요,H05,장소조건 해당 응답 있음,보류: 현장 근거 및 나머지 기준 미검증,False,W001,"장소조건에 해당하는 응답이 있고 예외 적용 요청은 없지만, 화재감시자가 미배치로 입..."
1,이후,배치,"배치 대상 검토: 장소조건 해당, 예외 적용 요청 없음",Q004,-,R002,배치 응답 확인,H05,장소조건 해당 응답 있음,보류: 현장 근거 및 나머지 기준 미검증,False,W001,장소조건에 해당하는 응답과 배치 응답이 확인되었습니다. 이 결과만으로 전체 안전조치...


## 내가 저장한 평가 두 건 비교
같은 현장·작업의 평가ID 두 개를 입력하세요. 원래 평가를 수정하지 않습니다. 계산 데이터나 엔진 버전, 분모가 다르면 점수차이 표시를 보류하고 두 기록을 나란히 보여줍니다.


In [42]:
before_evaluation_id = ""  # 이전 평가ID
after_evaluation_id = ""   # 이후 평가ID

if not before_evaluation_id.strip() or not after_evaluation_id.strip():
    print("저장된 평가 목록에서 같은 현장·작업의 평가ID 두 개를 입력하세요.")
else:
    try:
        comparison = engine.compare_evaluations(
            before_evaluation_id.strip(), after_evaluation_id.strip(),
            project_dir / "outputs" / "evaluations_demo.sqlite3",
        )
    except (ValueError, KeyError, FileNotFoundError) as error:
        print("비교할 수 없습니다:", error)
    else:
        display(pd.DataFrame([comparison["비교안내"]]))
        for title in ("평가요약", "변경항목", "추가응답변경", "화재감시자검토"):
            print(title)
            display(comparison[title])


저장된 평가 목록에서 같은 현장·작업의 평가ID 두 개를 입력하세요.
